In [1]:
import json
import math
from pathlib import Path
from scipy.stats import norm

# ----------------------------
# Inputs
# ----------------------------

json_paths = [
    Path(r"master_outputs\square\stouffers_jsons\chain_nonzero_psd_scan12.json"),
    Path(r"master_outputs\square\stouffers_jsons\chain_nonzero_psd_scan13.json"),
    Path(r"master_outputs\square\stouffers_jsons\chain_nonzero_psd_scan14.json"),
]

CHAIN_TYPES = {"excitatory", "inhibitory"}

# ----------------------------
# Helper functions
# ----------------------------

def z_for_alt_from_onesided_p(p, alternative):
    """
    Convert a one-sided p-value into a signed z-score such that:
      +z = evidence FOR the stated alternative
      -z = evidence AGAINST the stated alternative
    """
    if alternative == "greater":
        return norm.isf(p)          # upper-tail inversion
    elif alternative == "less":
        return -norm.ppf(p)         # lower-tail inversion + sign flip
    else:
        raise ValueError("alternative must be 'greater' or 'less'")

def stouffer_combine(zs, ws):
    """
    Directional Stouffer combination.
    Assumes +Z supports the alternative.
    """
    Z = sum(w * z for w, z in zip(ws, zs)) / math.sqrt(sum(w * w for w in ws))
    p = norm.sf(Z)   # one-sided meta p-value
    return Z, p

records = []

for path in json_paths:
    with open(path, "r") as f:
        scan = json.load(f)

    for row in scan:
        if row["chain_type"] in CHAIN_TYPES:
            records.append({
                "chain": row["chain_type"],
                "z": z_for_alt_from_onesided_p(
                    row["p_value"],
                    row["alternative"]
                ),
                "n1": row["n_shared"],
                "n2": row["n_disjoint"],
            })

# ----------------------------
# Stouffer meta-analysis by chain type
# ----------------------------

for chain in CHAIN_TYPES:
    subset = [r for r in records if r["chain"] == chain]

    zs = [r["z"] for r in subset]

    # Effective sample size weights
    w_eff = [
        math.sqrt((r["n1"] * r["n2"]) / (r["n1"] + r["n2"]))
        for r in subset
    ]

    Z_eff, p_eff = stouffer_combine(zs, w_eff)

    # Total sample size weights
    w_tot = [
        math.sqrt(r["n1"] + r["n2"])
        for r in subset
    ]

    Z_tot, p_tot = stouffer_combine(zs, w_tot)

    # ----------------------------
    # Report
    # ----------------------------

    print("\n~~~~~~~~~~~~~~~~~~~~~~~~~~~~~\n")
    print(f"{chain.upper()} — Stouffer's (directional, effective N weights)")
    print("w_i = sqrt[(n1 * n2) / (n1 + n2)]")
    print(f"Z_meta: {Z_eff}")
    print(f"p_meta: {p_eff}")



~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

EXCITATORY — Stouffer's (directional, effective N weights)
w_i = sqrt[(n1 * n2) / (n1 + n2)]
Z_meta: 0.8411240437749503
p_meta: 0.20013922333156603

~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

INHIBITORY — Stouffer's (directional, effective N weights)
w_i = sqrt[(n1 * n2) / (n1 + n2)]
Z_meta: 2.136467367721757
p_meta: 0.016320669029395805
